In [ ]:
import requests
import pandas as pd
import re
import time
from datetime import datetime, timedelta, timezone
from bs4 import BeautifulSoup

session = requests.Session()
session.headers.update({
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 Chrome/120.0.0.0 Safari/537.36",
    "Referer": "https://www.ogimet.com/metars.phtml.en",
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8",
    "Accept-Language": "en-US,en;q=0.5",
})

def init_session():
    try:
        session.get("https://www.ogimet.com/metars.phtml.en", timeout=60)
        time.sleep(3)
        print("Session initialised")
    except Exception as e:
        print(f"Could not init session: {e}")

def fetch_metars(icao, start_date, end_date, max_retries=5):
    url = "https://ogimet.com/display_metars2.php"
    params = {
        "lang": "en",
        "lugar": icao,
        "tipo": "ALL",
        "ord": "REV",
        "nil": "SI",
        "fmt": "html",
        "ano":  start_date.strftime("%Y"),
        "mes":  start_date.strftime("%m"),
        "day":  start_date.strftime("%d"),
        "hora": start_date.strftime("%H"),
        "anof": end_date.strftime("%Y"),
        "mesf": end_date.strftime("%m"),
        "dayf": end_date.strftime("%d"),
        "horaf": end_date.strftime("%H"),
        "minf": end_date.strftime("%M"),
        "send": "send"
    }

    for attempt in range(1, max_retries + 1):
        try:
            print(f"  Attempt {attempt}/{max_retries}...")
            r = session.get(url, params=params, timeout=90)  
            r.encoding = "utf-8"

            soup = BeautifulSoup(r.text, "html.parser")
            full_text = soup.get_text(separator="\n")

            if "METAR" in full_text:
                lines = []
                for line in full_text.split("\n"):
                    line = line.strip()
                    if re.search(rf'\b{icao}\b.*\d{{6}}Z', line):
                        lines.append(line)
                return lines
            else:
                print(f" No METAR in response on attempt {attempt}. Snippet: {full_text[:200]}")
                # Wait longer before retrying
                time.sleep(30 * attempt)

        except requests.exceptions.Timeout:
            print(f"  Timeout on attempt {attempt}. Waiting {30 * attempt}s before retry...")
            time.sleep(30 * attempt)  # 30s, 60s, 90s, 120s, 150s

        except requests.exceptions.ConnectionError as e:
            print(f"  Connection error on attempt {attempt}: {e}. Waiting 60s...")
            time.sleep(60)

        except Exception as e:
            print(f"  ❗ Unexpected error on attempt {attempt}: {e}")
            time.sleep(30)

    print(f"  All {max_retries} attempts failed.")
    return []


def parse_valid(metar, year, month):
    m = re.search(r"\b(\d{2})(\d{2})(\d{2})Z\b", metar)
    if m:
        day, hour, minute = int(m.group(1)), int(m.group(2)), int(m.group(3))
        try:
            dt_utc = datetime(year, month, day, hour, minute, tzinfo=timezone.utc)
            dt_ist = dt_utc.astimezone(timezone(timedelta(hours=5, minutes=30)))
            return dt_ist.replace(tzinfo=None)
        except ValueError:
            return None
    return None



def save_checkpoint(records, current):
    df = pd.DataFrame(records)
    df.to_excel("vidp_metars_checkpoint.xlsx", index=False)
    print(f"  Checkpoint saved ({len(records)} records, up to {current.strftime('%Y-%m')})")



icao = "VIDP"
start = datetime(2020, 7, 1) 
end = datetime(2026, 1, 31)

print("Initialising session...")
init_session()

all_records = []
current = start
months_done = 0

while current <= end:
    next_month = (current.replace(day=28) + timedelta(days=4)).replace(day=1)
    month_start = datetime(current.year, current.month, 1, 16, 0)
    last_day = (next_month - timedelta(days=1)).day
    month_end = datetime(current.year, current.month, last_day, 16, 59)

    print(f"\nFetching {icao} METARs from {month_start} to {month_end}...")
    metars = fetch_metars(icao, month_start, month_end)

    if not metars:
        print(f" No data for {current.strftime('%Y-%m')} — skipping.")
    else:
        print(f" Got {len(metars)} records for {current.strftime('%Y-%m')}")
        for m in metars:
            ts = parse_valid(m, current.year, current.month)
            all_records.append({"valid": ts, "metar": m})

    months_done += 1

    
    if months_done % 6 == 0:
        save_checkpoint(all_records, current)

    
    print(f"  Waiting 15s before next request...")
    time.sleep(15)
    current = next_month


df = pd.DataFrame(all_records)
df.to_excel("vidp_metars_2020_2026.xlsx", index=False)
print("\nFinished! Final file saved.")
print(df.head())

Initialising session...
✅ Session initialised

Fetching VIDP METARs from 2020-07-01 16:00:00 to 2020-07-31 16:59:00...
  Attempt 1/5...
✅ Got 1767 records for 2020-07
  Waiting 15s before next request...

Fetching VIDP METARs from 2020-08-01 16:00:00 to 2020-08-31 16:59:00...
  Attempt 1/5...
✅ Got 1549 records for 2020-08
  Waiting 15s before next request...

Fetching VIDP METARs from 2020-09-01 16:00:00 to 2020-09-30 16:59:00...
  Attempt 1/5...
✅ Got 1740 records for 2020-09
  Waiting 15s before next request...

Fetching VIDP METARs from 2020-10-01 16:00:00 to 2020-10-31 16:59:00...
  Attempt 1/5...
✅ Got 1774 records for 2020-10
  Waiting 15s before next request...

Fetching VIDP METARs from 2020-11-01 16:00:00 to 2020-11-30 16:59:00...
  Attempt 1/5...
✅ Got 1694 records for 2020-11
  Waiting 15s before next request...

Fetching VIDP METARs from 2020-12-01 16:00:00 to 2020-12-31 16:59:00...
  Attempt 1/5...
✅ Got 1782 records for 2020-12
  💾 Checkpoint saved (10306 records, up to 

In [ ]:

import pandas as pd
import numpy as np
import re
import lightgbm as lgb
import joblib
from pathlib import Path


DATA_PATH = r"C:\Users\ashna\OneDrive\Documents\COLLEGE\SEM 6\Minor Project 2\END SEM\vidp_metars_2020_2026.xlsx"
MODEL_DIR = Path("models_12h")
MODEL_DIR.mkdir(exist_ok=True)


SCALE = 1.018
OFFSET = 1  


def parse_metar(metar):
    out = {"vis": np.nan, "wind": 0, "temp": np.nan,
           "dew": np.nan, "fog": 0, "mist": 0}

    m = re.search(r"\b(\d{4})\b", metar)
    if m:
        out["vis"] = int(m.group(1))

    m = re.search(r"(\d{2,3})KT", metar)
    if m:
        out["wind"] = int(m.group(1))

    m = re.search(r"(M?\d{2})/(M?\d{2})", metar)
    if m:
        out["temp"] = int(m.group(1).replace("M", "-"))
        out["dew"] = int(m.group(2).replace("M", "-"))

    out["fog"] = int("FG" in metar)
    out["mist"] = int(("BR" in metar) or ("MIFG" in metar))

    return out


df = pd.read_excel(DATA_PATH, engine="openpyxl")
df.columns = df.columns.str.lower().str.strip()
df["valid"] = pd.to_datetime(df["valid"], dayfirst=True)

parsed = df["metar"].apply(parse_metar).apply(pd.Series)
df = pd.concat([df, parsed], axis=1)

df = df.dropna(subset=["vis"]).sort_values("valid")


df["hour"] = df["valid"].dt.hour
df["month"] = df["valid"].dt.month
df["night"] = ((df["hour"] >= 18) | (df["hour"] <= 6)).astype(int)
df["winter"] = df["month"].isin([11,12,1,2]).astype(int)


for l in range(1, 13):
    if l % 3 == 0:
        df[f"vis_lag_{l}"] = df["vis"].shift(l+1)
    else:
        df[f"vis_lag_{l}"] = df["vis"].shift(l)


for col in [f"vis_lag_{l}" for l in range(1, 13)]:
    df[col] = df[col] * SCALE

df = df.dropna()

FEATURES = [
    "wind", "temp", "dew", "fog", "mist",
    "hour", "month", "night", "winter"
] + [f"vis_lag_{l}" for l in range(1, 13)]


for step in range(1, 25):
    df[f"target_{step}"] = df["vis"].shift(-(step + OFFSET))


for step in range(1, 25):
    train_df = df.dropna(subset=[f"target_{step}"])

    X = train_df[FEATURES]
    y = train_df[f"target_{step}"]

    model = lgb.LGBMRegressor(
        n_estimators=800,
        learning_rate=0.03,
        num_leaves=64,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=42
    )

    model.fit(X, y)
    joblib.dump(model, MODEL_DIR / f"vis_t+{step}.pkl")

    print(f"Step {step} trained")

print("Training complete")

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.018524 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 2613
[LightGBM] [Info] Number of data points in the train set: 115347, number of used features: 21
[LightGBM] [Info] Start training from score 2628.770336
✅ Step 1 trained
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.019024 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 2613
[LightGBM] [Info] Number of data points in the train set: 115346, number of used features: 21
[LightGBM] [Info] Start training from score 2628.767118
✅ Step 2 trained
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.017457 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 2613
[LightGBM] [Info] Number of data points in the train set: 115345, number of us